In [8]:
import torch
bs = 2
c = 4
g = 2
h, w = 1, 1
a = torch.randn(bs * g, c//g, 1, 1)  # 形状是 [bs * g, 1, h, w]

print(a)
# b = a.view(bs,g,1,h,w)
# # print(b)
# b = b.squeeze(2)
# print('b:',b)

# 通过 view 转换形状
a_reshaped = a.view(bs, g, c // g)  # 现在是 [bs, g, h, w]

print(a_reshaped)  # 输出: torch.Size([4, 8, 32, 32])

tensor([[[[ 0.4772]],

         [[-0.4958]]],


        [[[-1.8267]],

         [[-0.0981]]],


        [[[-1.4837]],

         [[ 1.2000]]],


        [[[ 1.2198]],

         [[ 1.3814]]]])
tensor([[[ 0.4772, -0.4958],
         [-1.8267, -0.0981]],

        [[-1.4837,  1.2000],
         [ 1.2198,  1.3814]]])


In [16]:
import torch.nn as nn
a = torch.randn(2,2,2,2)
# a = torch.tensor([
#     []
# ])
print(a)

norm = nn.BatchNorm2d(2)
print(norm(a))

tensor([[[[ 0.1358,  0.3177],
          [ 0.1736,  0.1047]],

         [[ 0.6140,  0.6812],
          [ 0.0053,  0.3502]]],


        [[[-0.5438, -0.6979],
          [-0.1915, -0.6252]],

         [[-0.4190,  1.5870],
          [-2.1985,  0.5286]]]])
tensor([[[[ 0.7952,  1.2748],
          [ 0.8950,  0.7132]],

         [[ 0.4536,  0.5184],
          [-0.1333,  0.1992]]],


        [[[-0.9966, -1.4027],
          [-0.0677, -1.2112]],

         [[-0.5424,  1.3916],
          [-2.2582,  0.3712]]]], grad_fn=<NativeBatchNormBackward0>)


In [18]:
a = torch.randn(2,2,2,2)
print(a)

b = torch.tensor([
    [10,10],
    [100,100]
])
print(b.shape)

print(a*b)

c = b.unsqueeze(-1).unsqueeze(-1)
print(a*c)

tensor([[[[ 0.3281,  0.8468],
          [ 0.8810,  0.7434]],

         [[-0.8081, -1.0256],
          [ 0.5346,  1.3592]]],


        [[[ 1.8420, -0.7706],
          [ 1.2046,  0.0947]],

         [[-1.0631,  0.2151],
          [-1.6638,  0.1623]]]])
torch.Size([2, 2])
tensor([[[[   3.2810,    8.4682],
          [  88.1048,   74.3352]],

         [[  -8.0815,  -10.2556],
          [  53.4631,  135.9173]]],


        [[[  18.4196,   -7.7059],
          [ 120.4606,    9.4739]],

         [[ -10.6311,    2.1515],
          [-166.3806,   16.2318]]]])
tensor([[[[   3.2810,    8.4682],
          [   8.8105,    7.4335]],

         [[  -8.0815,  -10.2556],
          [   5.3463,   13.5917]]],


        [[[ 184.1956,  -77.0591],
          [ 120.4606,    9.4739]],

         [[-106.3114,   21.5148],
          [-166.3806,   16.2318]]]])


In [34]:
a = torch.randn(1,6,2,2) 
print(a)

a = a.view(1,2,3,2,2)
print(a.shape)

tensor([[[[ 0.1893, -0.7606],
          [-0.8121, -1.5294]],

         [[ 0.7072,  2.3621],
          [-0.6090, -1.1308]],

         [[ 0.7149, -2.5526],
          [ 0.5789,  0.5620]],

         [[ 0.0548, -0.0637],
          [ 1.3862,  1.4109]],

         [[-1.3158,  0.6758],
          [-1.1916,  0.8085]],

         [[ 1.3073,  0.5998],
          [ 1.7024, -0.5866]]]])
torch.Size([1, 2, 3, 2, 2])


In [36]:
b = torch.tensor(
    [[
        [
            [0.1,0.1],
            [0.1,0.1],
            ],
        [
            [100,100],
            [100,100]
        ]
    ]]
)
print(b.shape)
# print(a*b)
b = b.unsqueeze(2)
print(b)
print(b.shape)

torch.Size([1, 2, 2, 2])
tensor([[[[[  0.1000,   0.1000],
           [  0.1000,   0.1000]]],


         [[[100.0000, 100.0000],
           [100.0000, 100.0000]]]]])
torch.Size([1, 2, 1, 2, 2])


In [37]:
print(a*b)

tensor([[[[[ 1.8933e-02, -7.6059e-02],
           [-8.1205e-02, -1.5294e-01]],

          [[ 7.0716e-02,  2.3621e-01],
           [-6.0895e-02, -1.1308e-01]],

          [[ 7.1494e-02, -2.5526e-01],
           [ 5.7887e-02,  5.6197e-02]]],


         [[[ 5.4791e+00, -6.3652e+00],
           [ 1.3862e+02,  1.4109e+02]],

          [[-1.3158e+02,  6.7580e+01],
           [-1.1916e+02,  8.0848e+01]],

          [[ 1.3073e+02,  5.9976e+01],
           [ 1.7024e+02, -5.8660e+01]]]]])


In [38]:
import torch
import torch.nn as nn

class PartEnhancer(nn.Module):
    def __init__(self, in_channels, groups=8, reduction=16, use_fc=False):
        super(PartEnhancer, self).__init__()
        assert in_channels % groups == 0, "in_channels must be divisible by groups"
        self.groups = groups
        self.group_channels = in_channels // groups  # 每组的通道数
        self.use_fc = use_fc

        # **分组空间注意力**
        if use_fc:
            # 1x1 Conv + FC（更强表达能力）
            self.spatial_att = nn.Sequential(
                nn.Conv2d(in_channels, in_channels // 4, kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels // 4, groups, kernel_size=1)
            )
        else:
            # 只有 1x1 Conv（更高效）
            self.spatial_att = nn.Conv2d(in_channels, groups, kernel_size=1, stride=1, padding=0)

        # **通道注意力**
        self.avg_pool = nn.AdaptiveAvgPool2d(1)  # Global Avg Pooling
        self.channel_fc1 = nn.Linear(in_channels, in_channels // reduction)  # 降维
        self.relu = nn.ReLU(inplace=True)
        self.channel_fc2 = nn.Linear(in_channels // reduction, in_channels)  # 还原通道数
        
        # **Sigmoid 归一化**
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        B, C, H, W = x.shape
        g = self.groups
        c_per_group = C // g  # 每组通道数

        # **1️⃣ 分组空间注意力**
        att_map = self.spatial_att(x)  # [B, g, H, W]
        middle_result = att_map
        att_map = self.sigmoid(att_map)  # 归一化到 [0,1]

        # **2️⃣ 适配维度**
        att_map = att_map.view(B, g, 1, H, W)  # [B, g, 1, H, W]
        x_grouped = x.view(B, g, c_per_group, H, W)  # [B, g, c/g, H, W]

        # **3️⃣ 逐组加权**
        out = x_grouped * att_map  # [B, g, c/g, H, W] * [B, g, 1, H, W]
        out = out.view(B, C, H, W)  # 变回 [B, C, H, W]

        # **4️⃣ 通道注意力**
        avg_pooled = self.avg_pool(out).view(B, C)  # [B, C, 1, 1] -> [B, C]
        channel_weight = self.channel_fc1(avg_pooled)  # [B, C] -> [B, C//r]
        channel_weight = self.relu(channel_weight)  
        channel_weight = self.channel_fc2(channel_weight)  # [B, C//r] -> [B, C]
        channel_weight = self.sigmoid(channel_weight).view(B, C, 1, 1)  # [B, C] -> [B, C, 1, 1]

        # **5️⃣ 作用通道注意力**
        out = out * channel_weight  # [B, C, H, W] * [B, C, 1, 1]

        return out, middle_result




# **测试**
B, C, H, W = 2, 64, 32, 32  # 2 个 batch，64 通道，32x32 分辨率
x = torch.randn(B, C, H, W)  # 随机输入特征图

# 方案 1：只用 1x1 Conv
gca1 = GroupedChannelSpatialAttention(C, groups=8, reduction=16, use_fc=False)
out1 = gca1(x)

# 方案 2：1x1 Conv + FC
gca2 = GroupedChannelSpatialAttention(C, groups=8, reduction=16, use_fc=True)
out2 = gca2(x)

print(out1.shape)  # 期望输出: [B, C, H, W]
print(out2.shape)  # 期望输出: [B, C, H, W]


torch.Size([2, 64, 32, 32])


In [44]:
import torch
import torch.nn as nn

class GlobalEnhancer(nn.Module):
    def __init__(self, g):
        super(GlobalEnhancer, self).__init__()
        self.conv = nn.Conv2d(g, 1, kernel_size=1, bias=False)  # 1x1 conv 进行组间融合
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, feature_map):
        """
        x: [bs, g, h, w] - 组注意力图
        feature_map: [bs, c, h, w] - 原始特征图
        """
        attn = self.conv(x)  # [bs, g, h, w] -> [bs, 1, h, w]
        attn = self.sigmoid(attn)  # 归一化注意力权重
        output = feature_map * attn  # 注意力加权 [bs, c, h, w]
        return output  # 返回加权后的特征图 和 注意力权重




In [ ]:
class MPE(nn.Module):
    def __init__(self,in_channels, groups=8, reduction=8, use_fc=False):
        super(MPE, self).__init__()
        self.part_enhancer = PartEnhancer(in_channels, groups=groups, reduction=reduction, use_fc=use_fc)
        self.global_enhancher = GlobalEnhancer(g=groups)
                
    def forward(self,x):
        
        out, spatial_att = self.part_enhancer(x)
        out = self.global_enhancer(spatial_att,out)
        
        return out
    
    
# **测试**
B, C, H, W = 2, 64, 32, 32  # 2 个 batch，64 通道，32x32 分辨率
x = torch.randn(B, C, H, W)  # 随机输入特征图

# 方案 1：只用 1x1 Conv
# gca1 = GroupedChannelSpatialAttention(C, groups=8, reduction=16, use_fc=False)
# out1 = gca1(x)

# # 方案 2：1x1 Conv + FC
# gca2 = GroupedChannelSpatialAttention(C, groups=8, reduction=16, use_fc=True)
# out2 = gca2(x)

model = MPE(C, groups=8,reduction=8)
out = model(x)

print(out1.shape)  # 期望输出: [B, C, H, W]
print(out2.shape)  # 期望输出: [B, C, H, W]